# OData Query Validator — v1 Demo
Two checks: **empty query**, **$top limit**

In [23]:
import sys
sys.path.insert(0, '..')

from validation.odata_query_validator_v1 import ODataQueryValidator
from validation import ValidatorConfig

validator = ODataQueryValidator(config=ValidatorConfig.for_production())
print('Validator ready  |  max $top:', validator.config.max_retrieval_limit)

Validator ready  |  max $top: 500


In [24]:
def show(label, result):
    status = 'VALID' if result.is_valid else 'INVALID'
    print(f'[{status}]  {label}')
    for issue in result.issues:
        icon = {'critical': '', 'warning': '', 'info': ''}.get(issue.severity.value, '')
        print(f'   {icon}  {issue.message}')
    if result.estimated_row_count is not None:
        print(f'   → estimated rows: {result.estimated_row_count}')
    print()

## 1 — Valid queries

In [25]:
show('Basic select',        validator.validate('Users?$top=10&$select=Name,Email'))
show('With filter',         validator.validate('Orders?$top=50&$filter=TotalAmount gt 100&$orderby=OrderDate desc'))
show('With expand',         validator.validate('Customers?$top=20&$expand=Orders'))

[VALID]  Basic select
   → estimated rows: 10

[VALID]  With filter
   → estimated rows: 50

[VALID]  With expand
   → estimated rows: 20



## 2 — Empty query

In [26]:
show('Empty string',        validator.validate(''))
show('Whitespace only',     validator.validate('   '))
show('None',                validator.validate(None))

[INVALID]  Empty string
     No OData query was generated

[INVALID]  Whitespace only
     No OData query was generated

[INVALID]  None
     No OData query was generated



## 3 — $top limit enforcement

In [27]:
show('$top=10   (fine)',    validator.validate('Users?$top=10'))
show('$top=300  (warning)', validator.validate('Users?$top=300'))
show('$top=5000 (exceeds)', validator.validate('Users?$top=5000'))
show('$top=-1   (invalid)', validator.validate('Users?$top=-1'))
show('$top=abc  (invalid)', validator.validate('Users?$top=abc'))
show('missing $top',        validator.validate('Users?$filter=Age gt 18'))

[VALID]  $top=10   (fine)
   → estimated rows: 10

[VALID]  $top=300  (warning)
     $top value 300 is high (threshold: 250)
   → estimated rows: 300

[INVALID]  $top=5000 (exceeds)
     $top value 5000 exceeds maximum allowed: 500
   → estimated rows: 5000

[INVALID]  $top=-1   (invalid)
     Invalid $top value: -1 (must be positive)
   → estimated rows: -1

[INVALID]  $top=abc  (invalid)
     Invalid $top value: 'abc' (must be an integer)

[VALID]  missing $top
     Query does not specify $top limit



## 4 — ValidationResult: full structure returned to the module

In [29]:
import json

# Two representative cases: one valid, one invalid
valid_result   = validator.validate('Users?$top=10&$select=Name,Email')
invalid_result = validator.validate('Users?$top=5000')

for label, result in [('VALID query', valid_result), ('INVALID query', invalid_result)]:
    print(f'── {label} ──────────────────────────────')
    print(f'  is_valid              : {result.is_valid}')
    print(f'  has_critical_issues   : {result.has_critical_issues}')
    print(f'  has_warnings          : {result.has_warnings}')
    print(f'  estimated_row_count   : {result.estimated_row_count}')
    print(f'  get_error_message()   : {result.get_error_message()}')
    print(f'  validation_metadata   : {result.validation_metadata}')
    print(f'\n  to_dict():')
    print(json.dumps(result.to_dict(), indent=4))
    print()

── VALID query ──────────────────────────────
  is_valid              : True
  has_critical_issues   : False
  has_warnings          : False
  estimated_row_count   : 10
  get_error_message()   : None
  validation_metadata   : {'http_method': 'GET', 'parsed_params': ['$top', '$select']}

  to_dict():
{
    "is_valid": true,
    "query": "Users?$top=10&$select=Name,Email",
    "issues": [],
    "estimated_cost_score": null,
    "estimated_row_count": 10,
    "metadata": {
        "http_method": "GET",
        "parsed_params": [
            "$top",
            "$select"
        ]
    }
}

── INVALID query ──────────────────────────────
  is_valid              : False
  has_critical_issues   : True
  has_warnings          : False
  estimated_row_count   : 5000
  get_error_message()   : OData query validation failed:
  • $top value 5000 exceeds maximum allowed: 500
    Suggestion: Reduce $top to 500 or less
  validation_metadata   : {'http_method': 'GET', 'parsed_params': ['$top']}

  to_d

## 5 — State update 

Shows how `validate_query` in `nodes.py` reads and writes state.

In [30]:
def simulate_validate_query(state: dict) -> dict:
    """
    Mirrors the logic in nodes.py validate_query.
    Returns the partial state update that LangGraph applies.
    """
    odata_query = state.get("odata_query")

    if not odata_query:
        query_attempts = state.get("query_attempts", [])
        if query_attempts:
            query_attempts[-1]["validation_error"] = "No OData query was generated"
        return {
            "is_query_valid": False,
            "query_attempts": query_attempts,
            "error": "Query generation failed - no query produced",
        }

    result = validator.validate(query=odata_query, http_method="GET")

    query_attempts = state.get("query_attempts", [])
    if query_attempts:
        query_attempts[-1]["validation_error"] = (
            result.get_error_message() if not result.is_valid else None
        )

    return {
        "is_query_valid": result.is_valid,
        "query_attempts": query_attempts,
    }


def show_state_update(label, state):
    print(f'── {label} ──────────────────────────────')
    print(f'  STATE BEFORE:')
    print(f'    odata_query      : {state["odata_query"]}')
    print(f'    is_query_valid   : {state["is_query_valid"]}')
    print(f'    query_attempts   : {state["query_attempts"]}')

    update = simulate_validate_query(state)

    print(f'\n  STATE UPDATE (partial dict returned by node):')
    print(json.dumps(update, indent=4))

    # Apply update to show final state
    state.update(update)
    print(f'\n  STATE AFTER:')
    print(f'    is_query_valid   : {state["is_query_valid"]}')
    print(f'    query_attempts   : {state["query_attempts"]}')
    print()

In [31]:
show_state_update(
    'Query passes validation',
    {
        "odata_query"    : "Users?$top=10&$select=Name,Email",
        "is_query_valid" : False,
        "entity_schemas" : {},
        "retry_count"    : 0,
        "query_attempts" : [{"query": "Users?$top=10&$select=Name,Email", "validation_error": None, "attempt_number": 1}],
    }
)

show_state_update(
    'Query fails — $top exceeds limit',
    {
        "odata_query"    : "Users?$top=5000",
        "is_query_valid" : False,
        "entity_schemas" : {},
        "retry_count"    : 0,
        "query_attempts" : [{"query": "Users?$top=5000", "validation_error": None, "attempt_number": 1}],
    }
)

show_state_update(
    'Query fails — no query generated',
    {
        "odata_query"    : None,
        "is_query_valid" : False,
        "entity_schemas" : {},
        "retry_count"    : 0,
        "query_attempts" : [{"query": "", "validation_error": None, "attempt_number": 1}],
    }
)

── Query passes validation ──────────────────────────────
  STATE BEFORE:
    odata_query      : Users?$top=10&$select=Name,Email
    is_query_valid   : False
    query_attempts   : [{'query': 'Users?$top=10&$select=Name,Email', 'validation_error': None, 'attempt_number': 1}]

  STATE UPDATE (partial dict returned by node):
{
    "is_query_valid": true,
    "query_attempts": [
        {
            "query": "Users?$top=10&$select=Name,Email",
            "validation_error": null,
            "attempt_number": 1
        }
    ]
}

  STATE AFTER:
    is_query_valid   : True
    query_attempts   : [{'query': 'Users?$top=10&$select=Name,Email', 'validation_error': None, 'attempt_number': 1}]

── Query fails — $top exceeds limit ──────────────────────────────
  STATE BEFORE:
    odata_query      : Users?$top=5000
    is_query_valid   : False
    query_attempts   : [{'query': 'Users?$top=5000', 'validation_error': None, 'attempt_number': 1}]

  STATE UPDATE (partial dict returned by node):
{